<a href="https://colab.research.google.com/github/Saumyen10/PDF_Reader_RAG-app/blob/main/LangChain_PDF_Reader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Dependencies

In [ ]:
!pip install langchain
!pip install -U langchain-community
!pip install langchain-google-genai
!pip install faiss-cpu
!pip install pypdf
!pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is inc

Step 2: Add Gemini API

In [ ]:
import getpass
import os

# Set API key if not already present
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter API key for Gemini: ')

Enter API key for Gemini: ··········


Step 3: Select llm model

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    'gemini-3-flash-preview',
    model_provider='google_genai'
)

Step 4: import PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

#load the pdf
loader = PyPDFLoader("acsbr-016.pdf")

documents = loader.load()

#pdf becomes a list of documents

Step 5: Split Text into Chunks


LLM cannot process huge PDFs at once. So, we split text into smaller chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_split = RecursiveCharacterTextSplitter(
    chunk_size=1000,             #each chunk contains 1000 characters
    chunk_overlap=50            #50 characters overlap between chunks
)
#overlap: prevents loss of context

docs = text_split.split_documents(documents)

Step 6: Convert Chunks to Embeddings

Embeddings: Text -> Numerical vectors

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2"
)

Step 7: Store the vectors in FAISS Vector Database

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    docs,
    embeddings,
    # batch_size=50
)

Step 8: Question

In [ ]:
query = "What is this document about?"

Step 9: Retrieve Relevant chunks

Step A:

Query → embedding

Step B:

Compare with stored embeddings

Step C:

Return top 3 most relevant chunks

In [ ]:
doc_retrieve = vector_store.similarity_search(
    query,
    k=3
)

Step 10: create Prompt Template

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

doc_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

Step 11: Combine Retrieved Chunks


In [ ]:
context = "\n".join(
    [doc.page_content for doc in doc_retrieve]
)

Step 12: Generate Final Prompt

In [ ]:
final_prompt = doc_prompt.format(
    context=context,
    question=query
)

Step 13: Generate response from LLM

In [ ]:
response = llm.invoke(final_prompt)

print(response)

content=[{'type': 'text', 'text': 'Based on the provided context, the document is a statistical brief or report from the **U.S. Census Bureau** regarding **income-to-poverty ratios**.\n\nSpecifically, it discusses:\n*   **The distribution of people by income-to-poverty ratios** in Columbia, Puerto Rico, and metropolitan statistical areas (MSAs).\n*   **Data from the 2021 and 2022 American Community Survey (ACS)** 1-year estimates.\n*   **Technical definitions and methodology**, such as the definition of a metropolitan statistical area (a core urban area with a population of 50,000 or more), sampling variability, margins of error, and statistical significance levels.\n*   **Population exclusions**, noting that certain groups (like people in institutional group quarters, college dormitories, or military barracks) may be handled differently in the data.', 'extras': {'signature': 'EtYNCtMNAQw51sczfIjGFbyO28/QHpmP1Qm/PGO0QQUcjbtbMkV9heLFJChf3F1v0ZEJCcwmXJ5mD75kuS3+zV4gPpNlAVqQn/RKb9Pq/LtJH7

In [ ]:
print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, the document is a statistical brief or report from the **U.S. Census Bureau** regarding **income-to-poverty ratios**.\n\nSpecifically, it discusses:\n*   **The distribution of people by income-to-poverty ratios** in Columbia, Puerto Rico, and metropolitan statistical areas (MSAs).\n*   **Data from the 2021 and 2022 American Community Survey (ACS)** 1-year estimates.\n*   **Technical definitions and methodology**, such as the definition of a metropolitan statistical area (a core urban area with a population of 50,000 or more), sampling variability, margins of error, and statistical significance levels.\n*   **Population exclusions**, noting that certain groups (like people in institutional group quarters, college dormitories, or military barracks) may be handled differently in the data.', 'extras': {'signature': 'EtYNCtMNAQw51sczfIjGFbyO28/QHpmP1Qm/PGO0QQUcjbtbMkV9heLFJChf3F1v0ZEJCcwmXJ5mD75kuS3+zV4gPpNlAVqQn/RKb9Pq/LtJH7NpBgaMzT

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Loop Logic:

In [ ]:
while True:
  query  = input("\n Ask your question: ")

  if query.lower() == "exit":   #leave if you don't want to ask anymore question
    break


  doc_retrieve = vector_store.similarity_search(
      query,
      k=3
  )

  context = "\n\n".join(
      [doc.page_content for doc in doc_retrieve]
  )

  doc_prompt = PromptTemplate(
      input_variables=["context", "question"],
      template=prompt_template
  )

  final_prompt = doc_prompt.format(
      context=context,
      question=query
  )

  response = llm.invoke(final_prompt)

  print("\nAnswer:")

  print(response.text)


 Ask your question: What was the ACS national poverty rate in 2022?

Answer: 

In 2022, the ACS national poverty rate was **12.6 percent**.

 Ask your question: exit


**Structure:**


1. Load PDF
2. Split chunks
3. Create embeddings
4. Create FAISS
5. Create PromptTemplate

LOOP:

  1. Ask question
  2. Retrieve docs
  3. Format prompt
  4. Generate answer

## Complete Code:


In [ ]:
##Step 1: install dependencies


##Step 2: Add Gemini API key

import getpass
import os

# Set API key if not already present
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter API key for Gemini: ')


## Step 3: Select llm model

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    'gemini-3-flash-preview',
    model_provider='google_genai'
)


##Step 4: load PDF

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("acsbr-016.pdf") #load pdf
documents = loader.load()


## Step 5: Step 5: Split Text into Chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_split = RecursiveCharacterTextSplitter(
    chunk_size=1000,             #each chunk contains 1000 characters
    chunk_overlap=50            #50 characters overlap between chunks
)
docs = text_split.split_documents(documents)


## Step 6: Convert Chunks to Embeddings

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2"
)

## Step 7: Create FAISS vector database

from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    docs,
    embeddings,
    # batch_size=50
)


## Step 8: Create Prompt Template

prompt_template = """
  Use the following context to answer the question.

  Context:
  {context}

  Question:
  {question}

  Answer:
  """

#### LOOP:

while True:

  ## Step 9: Ask question
  query  = input("\n Ask your question: ")

  if query.lower() == "exit":   #leave if you don't want to ask anymore question
    break

  ## Step 10: Retrieve relevant docs
  doc_retrieve = vector_store.similarity_search(
      query,
      k=3
  )

   ## Step 11: Combine retrieved chunks
  context = "\n\n".join(
      [doc.page_content for doc in doc_retrieve]
  )

  ## Step 12: Format prompt
  doc_prompt = PromptTemplate(
      input_variables=["context", "question"],
      template=prompt_template
  )

  ## Step 12: Generate Final prompt
  final_prompt = doc_prompt.format(
      context=context,
      question=query
  )

 ## Step 14: Generate Response from llm
  response = llm.invoke(final_prompt)

  print("\nAnswer:")
  print(response.text)